In [1]:
import gymnasium as gym

import numpy as np
import time
import matplotlib
import matplotlib.pyplot as plt
import random 
import numpy as np
from collections import defaultdict
#import tensorflow as tf
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Input
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from collections import deque
 
 

env=gym.make('CartPole-v1')#,render_mode='human')

(state,_)=env.reset()

In [2]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),  # Increased size for hidden layers
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, states):
        input_list = np.array(states,dtype=np.float32)
        x=torch.from_numpy(input_list).float()
        return self.fc(x)
    

In [3]:
class DQAgent:

    def __init__(self, state_dim, action_dim, gamma=0.99, lr=0.001, batch_size=64, buffer_size=10000):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.memory = deque(maxlen=buffer_size)

        # Models
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.update_target_model()

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.loss_fn = nn.SmoothL1Loss() 
        
    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q_values = self.model(state_tensor)
            return torch.argmax(q_values).item()
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
            
    def train(self): 
        
        if len(self.memory) < self.batch_size:
            return  # Not enough samples to train

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)

        # Compute target Q-values
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            max_next_q_values = next_q_values.max(dim=1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        # Compute current Q-values
        q_values = self.model(states)
        current_q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Compute loss
        loss = self.loss_fn(current_q_values, target_q_values)

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        

## test

In [4]:
episodeNumber=30000
timeSteps=1000
episodesResults=[]
episodesTestResults=[]
episodesTestResultsDeviation=[]
numberOfTest=50
agent = DQAgent(4, 2)


In [5]:
for episodeIndex in range(1,episodeNumber+1):
    state,_=env.reset()
    
   
    for timeIndex in range(timeSteps+1):
        action=agent.get_action(state)
        next_state, reward, done, _, _ = env.step(action)
        agent.store_transition(state, action, reward, next_state, done)
        
        state=next_state
        agent.train()
        if (done or timeIndex==timeSteps-1 ):
            
            episodesResults.append(timeIndex)
            break
    
    if(episodeIndex %10 ==0):
        agent.update_target_model()
    if(episodeIndex %250 ==0):
        print("episode : ",episodeIndex)
    if(episodeIndex %500 ==0):
        tmp_epsilon=agent.epsilon
        agent.epsilon=0
        testResultList=[]
        for episodeIndex in range(numberOfTest):
            state,_=env.reset()
            for timeIndex in range(timeSteps):
                action=agent.get_action(tuple(state))
                next_state, reward, done, _, _ = env.step(action)
                state=next_state
                if (done):
                    testResultList.append(timeIndex)
                    break
        episodesTestResults.append(sum(testResultList)/numberOfTest)
        episodesTestResultsDeviation.append(np.std(episodesTestResults))
        agent.epsilon=tmp_epsilon

env.close()

C:\Users\mathi\AppData\Local\Temp\ipykernel_22044\2635204200.py:43: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  states = torch.FloatTensor(states)


episode :  250
episode :  500
episode :  750
episode :  1000
episode :  1250
episode :  1500
episode :  1750
episode :  2000
episode :  2250
episode :  2500
episode :  2750
episode :  3000
episode :  3250
episode :  3500
episode :  3750
episode :  4000
episode :  4250
episode :  4500
episode :  4750
episode :  5000
episode :  5250
episode :  5500
episode :  5750
episode :  6000
episode :  6250
episode :  6500
episode :  6750
episode :  7000
episode :  7250
episode :  7500
episode :  7750
episode :  8000
episode :  8250
episode :  8500
episode :  8750
episode :  9000
episode :  9250
episode :  9500
episode :  9750
episode :  10000
episode :  10250
episode :  10500
episode :  10750
episode :  11000
episode :  11250
episode :  11500
episode :  11750
episode :  12000
episode :  12250
episode :  12500
episode :  12750
episode :  13000
episode :  13250
episode :  13500
episode :  13750
episode :  14000
episode :  14250
episode :  14500
episode :  14750
episode :  15000
episode :  15250
episo

In [6]:
goal5LearnerEp=episodesResults
%store goal5LearnerEp

DQN1episodesTestResults=episodesTestResults
%store DQN1episodesTestResults

DQN1episodesTestResultsDeviation=episodesTestResultsDeviation
%store DQN1episodesTestResultsDeviation

Stored 'goal5LearnerEp' (list)
Stored 'DQN1episodesTestResults' (list)
Stored 'DQN1episodesTestResultsDeviation' (list)


In [7]:
print(episodesTestResultsDeviation)
print(episodesTestResults)
print(len(episodesResults))

[0.0, 47.92, 83.56643238898154, 85.33572288320994, 125.76863710798493, 115.4000818890524, 107.5950624009356, 107.88661637918764, 101.89246580417182, 96.66400206902259, 93.91111901356605, 89.91340494547455, 86.60044642327233, 86.69275916613068, 83.77845860495539, 83.88544420815808, 82.03079571945395, 81.60724510924865, 81.9473287766527, 80.02328821037037, 78.56414731052827, 78.18661234772374, 77.18043379187797, 77.19648694870915, 76.61295507784568, 75.56977677703024, 75.00323779750937, 74.24314912392076, 72.952192447366, 72.86351894390559, 72.38573037063536, 72.19934149359933, 71.5098294167589, 70.45145245690253, 70.31354665194688, 69.72077071530263, 68.95924646518421, 68.77547377370645, 68.70817627412696, 68.32333451318078, 67.50836852259863, 69.15050872000747, 68.53205680046621, 67.99945915408397, 67.55729862371152, 67.13763381228925, 66.98395093282643, 66.55022163227974, 67.12395471161256, 68.46913962479739, 68.17193766588352, 67.88863547368801, 67.54661592057082, 67.79109800804086, 